In [2]:
import pandas as pd
import os
from dotenv import load_dotenv
from sqlalchemy import create_engine, text
from sqlalchemy.exc import OperationalError, IntegrityError
from pandas.errors import DatabaseError
load_dotenv()

db_user = os.getenv("DB_USER")
db_password = os.getenv("DB_PASSWORD")
db_host = os.getenv("DB_HOST")
db_port = os.getenv("DB_PORT")
db_name = os.getenv("DB_NAME")

engine = create_engine(f"postgresql://{db_user}:{db_password}@{db_host}:{db_port}/{db_name}")

In [3]:

        
def extract(customers_path, orders_path):
    try:
        customers_df = pd.read_csv(customers_path)
    except FileNotFoundError:
        print('Customers file not found')
        customers_df = None
    try:   
            orders_df = pd.read_csv(orders_path)
    except FileNotFoundError:
            print('Orders file not found')
            orders_df = None
    return customers_df, orders_df

In [4]:
def clean(customers_df, orders_df):
    if customers_df is None or orders_df is None:
        return None, None, None
    else:
        customers_df['city'] = customers_df['city'].str.strip().str.title()
        customers_df['city'] = customers_df['city'].replace({"Ph": "Port Harcourt"})
        customers_df['customer_name'] = customers_df['customer_name'].str.strip().str.title()
        customers_df = customers_df.drop_duplicates()
        customers_df['email_missing'] = customers_df['email'].isnull()
        orders_df['unit_price'] = orders_df.groupby('product')['unit_price'].transform(lambda x: x.fillna(x.mean()))
        orphaned_orders = ~(orders_df['customer_id'].isin(customers_df['customer_id']))
        orphaned_orders_df = orders_df[orphaned_orders]
        print(f"{len(orphaned_orders_df)} orphaned orders rejected")
        orders_df = orders_df[~orphaned_orders]
        return customers_df, orders_df, orphaned_orders_df

In [5]:
def load(df, table_name, engine):
    try:
        with engine.connect() as conn:
                conn.execute(text(f"TRUNCATE TABLE {table_name}"))
                conn.commit()     
        df.to_sql(table_name, con=engine, if_exists="append", index=False)
        print(f'Loaded {len(df)} rows into {table_name}')
    except OperationalError:
        print('Database connection failed')
    except (IntegrityError, DatabaseError):
        print(f"{table_name} rows are duplicated")
    
    

In [6]:
customers_df, orders_df = extract(r'C:\Users\HP\Downloads\customers.csv', r'C:\Users\HP\Downloads\orders.csv')
customers_df, orders_df, orphaned_orders_df = clean(customers_df, orders_df)
load(orders_df, "orders", engine)
load(customers_df, "customers", engine)

5 orphaned orders rejected


Loaded 294 rows into orders


NotSupportedError: (psycopg2.errors.FeatureNotSupported) cannot truncate a table referenced in a foreign key constraint
DETAIL:  Table "orders" references "customers".
HINT:  Truncate table "orders" at the same time, or use TRUNCATE ... CASCADE.

[SQL: TRUNCATE TABLE customers]
(Background on this error at: https://sqlalche.me/e/20/tw8g)

In [ ]:
try:
    load(customers_df, "customers", engine)
except Exception as e:
    print(type(e))
    print(type(e).__mro__)

customers rows are duplicated


In [ ]:
customers_df = pd.read_csv(r'C:\Users\HP\Downloads\customers.csv')

In [ ]:
orders_df = pd.read_csv(r'C:\Users\HP\Downloads\orders.csv')

In [ ]:
customers_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 63 entries, 0 to 62
Data columns (total 5 columns):
 #   Column         Non-Null Count  Dtype
---  ------         --------------  -----
 0   customer_id    63 non-null     str  
 1   customer_name  63 non-null     str  
 2   email          60 non-null     str  
 3   city           63 non-null     str  
 4   signup_date    63 non-null     str  
dtypes: str(5)
memory usage: 2.6 KB


In [ ]:
customers_df.head()

,customer_id,customer_name,email,city,signup_date
0,CUST229,Ahmed Chukwu,ahmed.chukwu@gmail.com,Ibadan,2025-05-01
1,CUST222,Chinwe Danjuma,chinwe.danjuma@gmail.com,Kaduna,2024-04-16
2,CUST246,Tobi Nwachukwu,tobi.nwachukwu@gmail.com,Abuja,2025-09-14
3,CUST247,Bola Suleiman,bola.suleiman@gmail.com,enugu,2025-07-31
4,CUST234,Chidi Okoro,chidi.okoro@gmail.com,LAGOS,2025-01-23


In [ ]:
customers_df.isnull().sum()


customer_id      0
customer_name    0
email            3
city             0
signup_date      0
dtype: int64

In [ ]:
orders_df.head()

,order_id,customer_id,product,quantity,unit_price,order_date
0,ORD3269,CUST217,Water Bottle,4,12503.05,2024-12-06
1,ORD3241,CUST231,Notebook Set,1,22811.16,2024-10-24
2,ORD3279,CUST255,Notebook Set,2,17744.24,2025-07-15
3,ORD3177,CUST257,Notebook Set,2,42126.23,2025-01-28
4,ORD3203,CUST212,Notebook Set,3,NaN,2024-07-29


In [ ]:
orders_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 299 entries, 0 to 298
Data columns (total 6 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   order_id     299 non-null    str    
 1   customer_id  299 non-null    str    
 2   product      299 non-null    str    
 3   quantity     299 non-null    int64  
 4   unit_price   294 non-null    float64
 5   order_date   299 non-null    str    
dtypes: float64(1), int64(1), str(4)
memory usage: 14.1 KB


In [ ]:
orders_df.isnull().sum()

order_id       0
customer_id    0
product        0
quantity       0
unit_price     5
order_date     0
dtype: int64

In [ ]:
customers_df.duplicated().sum()

np.int64(3)

In [ ]:
orphaned_orders = ~(orders_df['customer_id'].isin(customers_df['customer_id']))

In [ ]:
orphaned_orders_df = orders_df[orphaned_orders]          

In [ ]:
orders_df = orders_df[~orphaned_orders]   

In [ ]:
print(len(orphaned_orders_df))
print(len(orders_df))

5
294


In [ ]:
customers_df = customers_df.drop_duplicates()

In [ ]:
print(customers_df.duplicated().sum())
len(customers_df)

0


60

In [ ]:
customers_df['email_missing'] = customers_df['email'].isnull()

In [ ]:
customers_df['email_missing'].sum()

np.int64(3)

In [ ]:
orders_df['unit_price'] = orders_df.groupby('product')['unit_price'].transform(lambda x: x.fillna(x.mean()))

In [ ]:
orders_df.isna().sum()

order_id       0
customer_id    0
product        0
quantity       0
unit_price     0
order_date     0
dtype: int64

In [ ]:
customers_df['customer_name'] = customers_df['customer_name'].str.strip().str.title()

In [ ]:
customers_df['city'] = customers_df['city'].str.strip().str.title()
print(customers_df['city'].unique())
customers_df['city'] = customers_df['city'].replace({"Ph": "Port Harcourt"})
print(customers_df['city'].unique())

<StringArray>
[       'Ibadan',        'Kaduna',         'Abuja',         'Enugu',
         'Lagos',          'Kano',            'Ph',    'Benin City',
 'Port Harcourt']
Length: 9, dtype: str
<StringArray>
[       'Ibadan',        'Kaduna',         'Abuja',         'Enugu',
         'Lagos',          'Kano', 'Port Harcourt',    'Benin City']
Length: 8, dtype: str


In [ ]:
import os
from dotenv import load_dotenv
from sqlalchemy import create_engine

load_dotenv()

db_user = os.getenv("DB_USER")
db_password = os.getenv("DB_PASSWORD")
db_host = os.getenv("DB_HOST")
db_port = os.getenv("DB_PORT")
db_name = os.getenv("DB_NAME")

engine = create_engine(f"postgresql://{db_user}:{db_password}@{db_host}:{db_port}/{db_name}")

In [ ]:
load(customers_df, 'customers', bad_engine)

Database connection failed


In [ ]:
with engine.connect() as conn:
    print("Connected successfully")

Connected successfully


In [ ]:
from sqlalchemy import text

with engine.connect() as conn:
    conn.execute(text("TRUNCATE TABLE orders, customers RESTART IDENTITY CASCADE"))
    conn.commit()

customers_df.to_sql('customers', engine, if_exists='append', index=False)
orders_df.to_sql('orders', engine, if_exists='append', index=False)

print("Load complete")

Load complete


In [ ]:
customers_df.head()

,customer_id,customer_name,email,city,signup_date,email_missing
0,CUST229,Ahmed Chukwu,ahmed.chukwu@gmail.com,Ibadan,2025-05-01,False
1,CUST222,Chinwe Danjuma,chinwe.danjuma@gmail.com,Kaduna,2024-04-16,False
2,CUST246,Tobi Nwachukwu,tobi.nwachukwu@gmail.com,Abuja,2025-09-14,False
3,CUST247,Bola Suleiman,bola.suleiman@gmail.com,Enugu,2025-07-31,False
4,CUST234,Chidi Okoro,chidi.okoro@gmail.com,Lagos,2025-01-23,False


In [ ]:
orders_df.head()

,order_id,customer_id,product,quantity,unit_price,order_date
0,ORD3269,CUST217,Water Bottle,4,12503.050000,2024-12-06
1,ORD3241,CUST231,Notebook Set,1,22811.160000,2024-10-24
2,ORD3279,CUST255,Notebook Set,2,17744.240000,2025-07-15
3,ORD3177,CUST257,Notebook Set,2,42126.230000,2025-01-28
4,ORD3203,CUST212,Notebook Set,3,24351.524118,2024-07-29


In [ ]:
orphaned_orders_df.head()

,order_id,customer_id,product,quantity,unit_price,order_date
88,ORD3299,CUST1003,Power Bank,2,40607.02,2025-04-16
102,ORD3298,CUST1002,Bluetooth Speaker,2,18998.54,2025-05-28
123,ORD3297,CUST1001,Water Bottle,3,34895.25,2025-06-29
149,ORD3295,CUST999,Wireless Mouse,3,13024.67,2024-10-04
252,ORD3296,CUST1000,Keyboard,2,18868.27,2025-07-17


In [ ]:
print(orders_df['customer_id'].isin(customers_df['customer_id']).all())

True


In [ ]:

def extract(customers_path, orders_path):
    try:
        customers_df = pd.read_csv(customers_path)
        orders_df = pd.read_csv(orders_path)
        return customers_df, orders_df
    except FileNotFoundError:
        print('File not found')
        return None, None

In [ ]:
c, o = extract('nonexistent_file.csv', 'nonexistent_file.csv')
print(c, o)

File not found
None None


In [ ]:
print(customers_df.head())
print(orders_df.head())

  customer_id   customer_name                     email    city signup_date
0     CUST229    Ahmed Chukwu    ahmed.chukwu@gmail.com  Ibadan  2025-05-01
1     CUST222  Chinwe Danjuma  chinwe.danjuma@gmail.com  Kaduna  2024-04-16
2     CUST246  Tobi Nwachukwu  tobi.nwachukwu@gmail.com   Abuja  2025-09-14
3     CUST247   Bola Suleiman   bola.suleiman@gmail.com   enugu  2025-07-31
4     CUST234     Chidi Okoro     chidi.okoro@gmail.com   LAGOS  2025-01-23
  order_id customer_id       product  quantity  unit_price  order_date
0  ORD3269     CUST217  Water Bottle         4    12503.05  2024-12-06
1  ORD3241     CUST231  Notebook Set         1    22811.16  2024-10-24
2  ORD3279     CUST255  Notebook Set         2    17744.24  2025-07-15
3  ORD3177     CUST257  Notebook Set         2    42126.23  2025-01-28
4  ORD3203     CUST212  Notebook Set         3         NaN  2024-07-29


In [ ]:
def clean(customers_df, orders_df):
    customers_df['city'] = customers_df['city'].str.strip().str.title()
    customers_df['city'] = customers_df['city'].replace({"Ph": "Port Harcourt"})
    customers_df['customer_name'] = customers_df['customer_name'].str.strip().str.title()
    customers_df = customers_df.drop_duplicates()
    customers_df['email_missing'] = customers_df['email'].isnull()
    orders_df['unit_price'] = orders_df.groupby('product')['unit_price'].transform(lambda x: x.fillna(x.mean()))
    orphaned_orders = ~(orders_df['customer_id'].isin(customers_df['customer_id']))
    orphaned_orders_df = orders_df[orphaned_orders] 
    orders_df = orders_df[~orphaned_orders]   
    return customers_df, orders_df, orphaned_orders_df

In [ ]:
customers_df, orders_df, orphaned_orders_df = clean(customers_df, orders_df)

In [ ]:
print(customers_df['city'].unique())
print(customers_df.isnull().sum())
print(len(orphaned_orders_df))

<StringArray>
[       'Ibadan',        'Kaduna',         'Abuja',         'Enugu',
         'Lagos',          'Kano', 'Port Harcourt',    'Benin City']
Length: 8, dtype: str
customer_id      0
customer_name    0
email            3
city             0
signup_date      0
email_missing    0
dtype: int64
5
